In [22]:
# Basic Import
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt 
import seaborn as sns
# Modelling
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score,roc_auc_score
from sklearn.model_selection import train_test_split


from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler,OneHotEncoder
import warnings

In [23]:
df=pd.read_csv('processed_data/integrated_cutomer_dataset.csv')

In [24]:
df.head()

,tenure_months,contract_type,monthly_spend,total_spend,order_count,avg_order_value,payment_method,autopay,num_products_owned,complaint_count,...,billing_cycle,autopay_enabled,paperless_billing,late_payments_12m,price_increase_last_year,discount_pct,payment_failures_90d,billing_address_city,tax_exempt,invoice_delivery
0,65,annual,132.82,9670.8,12,322.36,credit_card,0,3,2,...,No Plan,0.0,0.0,0.0,0.0,0.0,0.0,No Plan,0.0,No Plan
1,53,annual,286.50,22844.1,76,761.47,ach,1,3,2,...,annual,1.0,1.0,2.0,0.0,14.7,0.0,North Hollyfurt,0.0,mail
2,46,biennial,65.39,3891.6,41,129.72,invoice,0,5,2,...,No Plan,0.0,0.0,0.0,0.0,0.0,0.0,No Plan,0.0,No Plan
3,87,annual,55.78,3067.8,14,102.26,invoice,1,7,3,...,monthly,1.0,1.0,0.0,1.0,5.1,0.0,South Allisonview,0.0,mail
4,2,annual,269.11,21353.4,78,711.78,credit_card,0,1,1,...,monthly,1.0,0.0,1.0,0.0,21.1,0.0,Lake Craigburgh,0.0,mail


In [25]:
df.columns

Index(['tenure_months', 'contract_type', 'monthly_spend', 'total_spend',
       'order_count', 'avg_order_value', 'payment_method', 'autopay',
       'num_products_owned', 'complaint_count', 'satisfaction_score',
       'churned', 'churn_reason', 'predicted_churn_prob',
       'retention_offer_sent', 'offer_accepted', 'country', 'state', 'city',
       'age', 'gender', 'occupation', 'marital_status', 'household_size',
       'segment', 'lifetime_value', 'loyalty_points', 'preferred_channel',
       'preferred_language', 'account_status', 'referral_source',
       'email_opt_in', 'sms_opt_in', 'login_frequency_30d',
       'pages_per_session', 'avg_session_duration_min', 'mobile_app_usage_pct',
       'email_open_rate', 'email_click_rate', 'push_notification_opt_in',
       'nps_score', 'csat_score', 'feature_adoption_score',
       'community_forum_posts', 'referrals_made', 'support_escalations_90d',
       'product_tours_completed', 'wishlist_items', 'cart_abandonment_rate',
       'p

In [26]:
df.shape

(150000, 71)

In [27]:
df.drop(columns=[
    "churn_reason",              # Data leakage
    "predicted_churn_prob",      # Data leakage
    "plan_name",                 # ~20,000 unique values
    "billing_address_city",      # ~13,000 unique values
    "city",                      # ~13,000 unique values
    "occupation",                # 640 unique values
    "country",                   # 195 categories
    "state"                      # 60 categories
], inplace=True)

In [28]:
df.shape

(150000, 63)

In [29]:
X=df.drop("churned",axis=1)
y = df["churned"]

In [30]:
X.shape

(150000, 62)

In [31]:
num_cols=X.select_dtypes(include=['int64','Float64']).columns
cat_cols=X.select_dtypes(exclude=['int64','Float64']).columns

In [32]:
for col in cat_cols:
    print(col)
    print(X[col].nunique())
    print(X[col].unique())
    print("===============")
    

contract_type
3
<StringArray>
['annual', 'biennial', 'month-to-month']
Length: 3, dtype: str
payment_method
4
<StringArray>
['credit_card', 'ach', 'invoice', 'paypal']
Length: 4, dtype: str
retention_offer_sent
4
<StringArray>
['email', 'sms', 'call', 'none']
Length: 4, dtype: str
gender
4
<StringArray>
['Unknown', 'Other', 'M', 'F']
Length: 4, dtype: str
marital_status
5
<StringArray>
['Unknown', 'widowed', 'married', 'divorced', 'single']
Length: 5, dtype: str
segment
5
<StringArray>
['Unknown', 'Gold', 'Platinum', 'Silver', 'Bronze']
Length: 5, dtype: str
preferred_channel
5
<StringArray>
['Unknown', 'web', 'email', 'sms', 'app']
Length: 5, dtype: str
preferred_language
5
<StringArray>
['Unknown', 'de', 'es', 'fr', 'en']
Length: 5, dtype: str
account_status
4
<StringArray>
['Unknown', 'suspended', 'active', 'inactive']
Length: 4, dtype: str
referral_source
6
<StringArray>
['Unknown', 'paid_ads', 'partner', 'event', 'organic', 'referral']
Length: 6, dtype: str
engagement_tier
4
<Stri

In [33]:
preprocessor=ColumnTransformer(
    transformers=[
        ("Encoding",OneHotEncoder(handle_unknown="ignore"),cat_cols),
        ("Scaling",StandardScaler(),num_cols)
    ]
)

In [34]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)
X_train=preprocessor.fit_transform(X_train)
X_test=preprocessor.transform(X_test)

In [35]:
X_train.shape, X_test.shape

((120000, 111), (30000, 111))

In [36]:
def evaluate_metrics(true,pred):
    accuracy=accuracy_score(true,pred)
    precision=precision_score(true,pred)
    recall=recall_score(true,pred)
    f1=f1_score(true,pred)
    roc=roc_auc_score(true,pred)
    return (accuracy,precision,recall,f1,roc)


In [37]:
models={
    'logistic Regression':LogisticRegression(),
    'Random Forest':RandomForestClassifier(),
    'XG Boost':XGBClassifier(),
    'Light GBM':LGBMClassifier()
}
results = []

for name, model in models.items():

    print("=" * 50)
    print(f"Model: {name}")

    print("Training...")
    model.fit(X_train, y_train)
    print("Training Completed.")

    print("Testing...")
    y_pred = model.predict(X_test)
    print("Testing Completed.")

    acc, pre, rec, f1, roc = evaluate_metrics(y_test, y_pred)

    results.append({
        "Model": name,
        "Accuracy": acc,
        "Precision": pre,
        "Recall": rec,
        "F1-Score": f1,
        "ROC-AUC": roc
    })

    print("\nModel Performance")
    print(f"Accuracy : {acc:.4f}")
    print(f"Precision: {pre:.4f}")
    print(f"Recall   : {rec:.4f}")
    print(f"F1-Score : {f1:.4f}")
    print(f"ROC-AUC  : {roc:.4f}")



Model: logistic Regression
Training...
Training Completed.
Testing...
Testing Completed.

Model Performance
Accuracy : 0.8279
Precision: 0.8479
Recall   : 0.7327
F1-Score : 0.7861
ROC-AUC  : 0.8164
Model: Random Forest
Training...
Training Completed.
Testing...
Testing Completed.

Model Performance
Accuracy : 0.9267
Precision: 1.0000
Recall   : 0.8301
F1-Score : 0.9071
ROC-AUC  : 0.9150
Model: XG Boost
Training...
Training Completed.
Testing...
Testing Completed.

Model Performance
Accuracy : 0.9260
Precision: 0.9977
Recall   : 0.8304
F1-Score : 0.9064
ROC-AUC  : 0.9145
Model: Light GBM
Training...
[LightGBM] [Info] Number of positive: 51789, number of negative: 68211
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.023033 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 4660
[LightGBM] [Info] Number of data points in the train set: 12000

In [38]:
corr = df.corr(numeric_only=True)["churned"].sort_values(ascending=False)
print(corr)

churned                      1.000000
complaint_count              0.566990
total_spend                  0.294498
avg_order_value              0.294498
monthly_spend                0.294498
email_click_rate             0.004080
pages_per_session            0.003767
login_frequency_30d          0.003756
price_increase_last_year     0.003614
discount_pct                 0.002770
autopay_enabled              0.002637
payment_failures_90d         0.002421
recency_days                 0.001987
monetary                     0.001909
loyalty_points               0.001760
age                          0.001688
sms_opt_in                   0.001634
avg_session_duration_min     0.001469
csat_score                   0.001442
m_score                      0.001343
offer_accepted               0.001156
push_notification_opt_in     0.001028
frequency                    0.000982
num_products_owned           0.000841
referrals_made               0.000796
promo_code_usage_90d         0.000299
tax_exempt  

In [39]:
from sklearn.metrics import confusion_matrix

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print(confusion_matrix(y_test, y_pred))

[[15351  1702]
 [ 3461  9486]]
